In [1]:
from pathlib import Path
from typing import Dict, List, Set, Tuple
from docplex.mp.dvar import Var
from docplex.mp.model import Model

Arco = Tuple[str, str]


def ler_instancia_pfm(
    caminho: Path,
) -> Tuple[List[str], Dict[Arco, float], str, str]:
    linhas = [
        linha.strip()
        for linha in caminho.read_text(encoding="utf-8").splitlines()
        if linha.strip()
    ]
    partes = linhas[0].split()
    n_arcos = int(partes[1])
    origem = partes[2]
    destino = partes[3]

    nos_set: Set[str] = {origem, destino}
    arcos: Dict[Arco, float] = {}

    for linha in linhas[1 : 1 + n_arcos]:
        u, v, cap_str = linha.split()
        nos_set.update([u, v])
        arcos[(u, v)] = float(cap_str)

    return sorted(nos_set), arcos, origem, destino


nos, arcos, origem, destino = ler_instancia_pfm(Path("in.txt"))
print(
    f"Instância carregada: {len(nos)} nós e {len(arcos)} arcos "
    f"(origem: {origem}, destino: {destino})."
)

Instância carregada: 7 nós e 11 arcos (origem: S, destino: T).


In [2]:
mdl = Model(name="Problema_de_Fluxo_Maximo")

# Variáveis inteiras 0 <= x_ij <= u_ij (l_ij = 0)
x: Dict[Arco, Var] = {
    (u, v): mdl.integer_var(lb=0, ub=cap, name=f"x_{u}{v}")
    for (u, v), cap in arcos.items()
}

# (1) Função Objetivo: maximizar o fluxo total que sai da origem S
mdl.maximize(mdl.sum(x[(u, v)] for (u, v) in arcos if u == origem))

# (2) Conservação de fluxo nos nós intermediários (i != S e i != T)
for no in nos:
    if no in (origem, destino):
        continue
    fluxo_saida = mdl.sum(x[(u, v)] for (u, v) in arcos if u == no)
    fluxo_entrada = mdl.sum(x[(u, v)] for (u, v) in arcos if v == no)
    mdl.add_constraint(fluxo_saida == fluxo_entrada, ctname=f"fluxo_{no}")

print("Modelo PFM construído com sucesso.")

Modelo PFM construído com sucesso.


In [3]:
sol = mdl.solve(log_output=False)

if sol is None:
    print("Nenhuma solução encontrada.")
else:
    print(f"Status: {mdl.solve_details.status}")
    print(f"Função Objetivo (Fluxo Máximo {origem}->{destino}): {sol.objective_value:.2f}\n")
    print("Fluxo ótimo por arco:")
    for (u, v), cap in arcos.items():
        val = int(round(sol.get_value(x[(u, v)])))
        print(f"  x[{u}->{v}] = {val:>2d} / {cap:.0f}")

Status: integer optimal solution
Função Objetivo (Fluxo Máximo S->T): 19.00

Fluxo ótimo por arco:
  x[S->A] =  8 / 8
  x[S->D] = 11 / 18
  x[A->B] =  8 / 12
  x[A->C] =  0 / 4
  x[A->D] =  0 / 10
  x[B->T] = 13 / 20
  x[C->B] =  5 / 7
  x[C->E] =  0 / 14
  x[D->C] =  5 / 5
  x[D->E] =  6 / 10
  x[E->T] =  6 / 6
